# LangChain Evaluation on Langfuse Trace Data

This notebook demonstrates how to run LangChain's built-in LLM evaluators against traces already stored in Langfuse.

Source: https://langfuse.com/guides/cookbook/evaluation_with_langchain

**Approach:** Fetch past generations from Langfuse, run LangChain criteria evaluators on each output, and push the scores back as Langfuse trace scores — so all quality signals live alongside the original traces in the dashboard.

**Evaluation criteria covered:**
- `hallucination` — does the output contain information not present in the input?
- `conciseness`, `relevance`, `coherence`, `helpfulness`
- `harmfulness`, `maliciousness`, `controversiality`, `misogyny`, `criminality`, `insensitivity`

## 1. Install Dependencies

In [ ]:
%pip install langfuse langchain langchain-openai --upgrade

## 2. Configure Credentials

Set your Langfuse and OpenAI credentials. Langfuse project keys are available under Project Settings at https://cloud.langfuse.com.

Regional endpoints:
- EU: `https://cloud.langfuse.com`
- US: `https://us.cloud.langfuse.com`
- Japan: `https://jp.cloud.langfuse.com`
- HIPAA: `https://hipaa.cloud.langfuse.com`

In [ ]:
import os

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

## 3. Configure Evaluation Model and Criteria

We use `gpt-3.5-turbo-instruct` as the judge model — cheaper than GPT-4 and sufficient for binary criteria evaluation.

Toggle individual criteria on/off with the `EVAL_TYPES` dict. The hallucination check uses a `LabeledCriteriaEvalChain` (requires a reference), while all others use the simpler `criteria` evaluator.

In [ ]:
os.environ["EVAL_MODEL"] = "gpt-3.5-turbo-instruct"

EVAL_TYPES = {
    "hallucination": True,
    "conciseness": True,
    "relevance": True,
    "coherence": True,
    "harmfulness": True,
    "maliciousness": True,
    "helpfulness": True,
    "controversiality": True,
    "misogyny": True,
    "criminality": True,
    "insensitivity": True,
}

## 4. Initialize Langfuse Client

In [ ]:
from langfuse import get_client

langfuse = get_client()

if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

## 5. Fetch Trace Data from Langfuse

Paginate through the Langfuse trace list and collect all generations for a given user. Swap `user_id` for a `name` filter (or remove it) depending on how your traces are tagged.

In [ ]:
def fetch_all_pages(name=None, user_id=None, limit=50):
    page = 1
    all_data = []

    while True:
        response = langfuse.api.trace.list(
            name=name, limit=limit, user_id=user_id, page=page
        )
        if not response.data:
            break
        all_data.extend(response.data)
        page += 1

    return all_data

In [ ]:
generations = fetch_all_pages(user_id="user_123")
print(f"Fetched {len(generations)} traces.")

In [ ]:
# Inspect the first trace id
generations[0].id

## 6. Set Up LangChain Evaluators

Two evaluator factories:
- `get_evaluator_for_key` — generic LangChain `criteria` evaluator for any named criterion
- `get_hallucination_eval` — custom `LabeledCriteriaEvalChain` with a bespoke hallucination definition that checks whether the output introduces information not present in the input

In [ ]:
from langchain.evaluation import load_evaluator
from langchain_openai import OpenAI
from langchain.evaluation.criteria import LabeledCriteriaEvalChain


def get_evaluator_for_key(key: str):
    llm = OpenAI(temperature=0, model=os.environ.get("EVAL_MODEL"))
    return load_evaluator("criteria", criteria=key, llm=llm)


def get_hallucination_eval():
    criteria = {
        "hallucination": (
            "Does this submission contain information"
            " not present in the input or reference?"
        ),
    }
    llm = OpenAI(temperature=0, model=os.environ.get("EVAL_MODEL"))
    return LabeledCriteriaEvalChain.from_llm(llm=llm, criteria=criteria)

## 7. Execute Evaluation and Score Traces

### 7a. Standard Criteria Evaluation

For each trace and each enabled criterion (excluding hallucination), run the evaluator and push the score + reasoning back to Langfuse.

In [ ]:
def execute_eval_and_score():
    for generation in generations:
        criteria = [
            key
            for key, value in EVAL_TYPES.items()
            if value and key != "hallucination"
        ]

        for criterion in criteria:
            eval_result = get_evaluator_for_key(criterion).evaluate_strings(
                prediction=generation.output,
                input=generation.input,
            )
            print(eval_result)

            langfuse.create_score(
                name=criterion,
                trace_id=generation.id,
                observation_id=generation.id,
                value=eval_result["score"],
                comment=eval_result["reasoning"],
            )


execute_eval_and_score()

### 7b. Hallucination Evaluation

Hallucination uses a `LabeledCriteriaEvalChain` which requires both `input` and `reference`. Here we pass the input as its own reference — a practical default when no ground truth is available.

Scores are only pushed when both `score` and `reasoning` are non-null.

In [ ]:
def eval_hallucination():
    chain = get_hallucination_eval()

    for generation in generations:
        eval_result = chain.evaluate_strings(
            prediction=generation.output,
            input=generation.input,
            reference=generation.input,
        )
        print(eval_result)

        if (
            eval_result is not None
            and eval_result["score"] is not None
            and eval_result["reasoning"] is not None
        ):
            langfuse.create_score(
                name="hallucination",
                trace_id=generation.id,
                observation_id=generation.id,
                value=eval_result["score"],
                comment=eval_result["reasoning"],
            )

In [ ]:
if EVAL_TYPES.get("hallucination") is True:
    eval_hallucination()

## 8. Flush Pending Scores

The Langfuse SDK batches requests asynchronously. Call `flush()` to ensure all `create_score` calls have been sent before the notebook exits.

In [ ]:
langfuse.flush()
print("All scores flushed to Langfuse.")

---

## Summary

| Step | What happens |
|---|---|
| Fetch traces | Paginate Langfuse API to retrieve all generations for a user/name filter |
| Criteria eval | LangChain `load_evaluator("criteria")` scores each output on 10 named dimensions |
| Hallucination eval | `LabeledCriteriaEvalChain` checks whether the output introduces facts not in the input |
| Score push | `langfuse.create_score()` attaches each score + LLM reasoning to its source trace |
| Flush | `langfuse.flush()` drains the async send queue |

All scores are now visible in the Langfuse UI under each trace, filterable and chartable across your dataset.